# Análise do Monitor de Crédito e Risco do Brasil

Este notebook responde as cinco perguntas do projeto usando os dados já tratados no PostgreSQL (camada `analytics`). Cada pergunta segue o mesmo padrão: roda a query em SQL, mostra um gráfico, resume os números numa tabela menor e fecha com a leitura de negócio.

Fonte: Sistema Gerenciador de Séries Temporais (SGS) do Banco Central. Período de janeiro de 2010 até o mês mais recente disponível.

## Configuração

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
import matplotlib.pyplot as plt
from src.db import get_engine
from src.config import SQL_DIR

pd.set_option("display.max_rows", 12)
engine = get_engine()

COR = {
    "azul":     "#1B4F8A",
    "vermelho": "#C0392B",
    "verde":    "#1A7A4A",
    "roxo":     "#5B2C8D",
    "laranja":  "#D35400",
    "cinza":    "#7F8C8D",
}

plt.rcParams.update({
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        110,
})

def rodar(arquivo):
    """Le uma query de sql/perguntas/ e devolve o resultado como DataFrame."""
    sql = (SQL_DIR / "perguntas" / arquivo).read_text(encoding="utf-8")
    return pd.read_sql(sql, engine)

## Pergunta 1: a inadimplência está subindo ou caindo, e de quem?

Comparação entre a inadimplência da pessoa física (PF) e da pessoa jurídica (PJ) ao longo do tempo.

In [ ]:
p1 = rodar("p1_inadimplencia.sql")
p1["mes"] = pd.to_datetime(p1["mes"])

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(p1["mes"], p1["inadimplencia_pf"], color=COR["azul"],     label="Pessoa Física")
ax.plot(p1["mes"], p1["inadimplencia_pj"], color=COR["vermelho"], label="Pessoa Jurídica", linestyle="--")
ax.set_title("Inadimplência da Carteira de Crédito", fontsize=13)
ax.set_ylabel("% em atraso", fontsize=10)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}%")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
resumo_p1 = pd.DataFrame({
    "minimo": [p1["inadimplencia_pf"].min(), p1["inadimplencia_pj"].min()],
    "maximo": [p1["inadimplencia_pf"].max(), p1["inadimplencia_pj"].max()],
    "mais_recente": [p1["inadimplencia_pf"].dropna().iloc[-1],
                     p1["inadimplencia_pj"].dropna().iloc[-1]],
}, index=["PF", "PJ"]).round(2)
resumo_p1

### Conclusão

A inadimplência da pessoa física fica sempre acima da pessoa jurídica, em média uns 1,7 ponto percentual a mais. A PF veio de um pico de 5,5% em 2012, despencou até 2,8% no fim de 2020, no período de auxílio emergencial e crédito mais cauteloso, e voltou a subir desde então, batendo perto de 5,4% no começo de 2026. A PJ é mais baixa e mais comportada, com pico de 4,0% em 2017 e o mesmo fundo lá em 2020. A leitura é direta: quando o crédito aperta, quem sente primeiro é a pessoa física.

## Pergunta 2: as famílias estão mais endividadas?

Endividamento (quanto a família deve em relação à renda de um ano) e comprometimento de renda (fatia da renda mensal que já vai para pagar dívida).

In [ ]:
p2 = rodar("p2_endividamento.sql")
p2["mes"] = pd.to_datetime(p2["mes"])

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(p2["mes"], p2["endividamento_familias"], color=COR["azul"],     label="Endividamento (% renda anual)")
ax.plot(p2["mes"], p2["comprometimento_renda"],  color=COR["vermelho"], label="Comprometimento (% renda mensal)", linestyle="--")
ax.set_title("Endividamento e Comprometimento de Renda das Famílias", fontsize=13)
ax.set_ylabel("%", fontsize=10)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}%")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
corr_p2 = p2["endividamento_familias"].corr(p2["comprometimento_renda"])
print(f"Correlacao endividamento x comprometimento: {corr_p2:.2f}")

p2[["endividamento_familias", "comprometimento_renda"]].describe().round(1)

### Conclusão

Endividamento e comprometimento de renda caminham praticamente juntos, com correlação de 0,81. O endividamento das famílias saiu de uns 30% no começo de 2010 e chegou a quase 50% em meados de 2022, o maior nível da série. O comprometimento de renda subiu de 22% para perto de 29%. Ou seja, as famílias passaram a dever mais e, ao mesmo tempo, a gastar um pedaço maior do que ganham só para dar conta dessas dívidas.

## Pergunta 3: o spread acompanha os juros?

O spread bancário é a diferença entre o juro que o banco cobra e o custo de captação dele. A pergunta é se ele segue a Selic ou se descola. Para enxergar isso ao longo do tempo, a query traz também a correlação móvel de 12 meses entre os dois.

In [ ]:
p3 = rodar("p3_spread.sql")
p3["mes"] = pd.to_datetime(p3["mes"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(p3["mes"], p3["spread_total"], color=COR["azul"],     label="Spread (% a.a.)")
ax1.plot(p3["mes"], p3["selic_meta"],   color=COR["vermelho"], label="Selic (% a.a.)", linestyle="--")
ax1.set_title("Spread Bancário e Selic", fontsize=13)
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:.1f}%")
ax1.legend(fontsize=9)

corr = p3["corr_movel_12m"]
ax2.plot(p3["mes"], corr, color=COR["roxo"], linewidth=1.8)
ax2.axhline(0, color=COR["cinza"], linewidth=0.8)
ax2.fill_between(p3["mes"], corr, 0, where=(corr > 0), alpha=0.10, color=COR["azul"])
ax2.fill_between(p3["mes"], corr, 0, where=(corr < 0), alpha=0.10, color=COR["cinza"])
ax2.set_title("Correlação Móvel 12 Meses — Spread × Selic", fontsize=13)
ax2.set_ylabel("Correlação (r)", fontsize=10)

fig.tight_layout(h_pad=2)
plt.show()

In [ ]:
corr_geral = p3["spread_total"].corr(p3["selic_meta"])
meses_neg = int((p3["corr_movel_12m"] < 0).sum())
print(f"Correlacao geral spread x Selic: {corr_geral:.2f}")
print(f"Meses com correlacao movel negativa: {meses_neg}")

### Conclusão

O spread acompanha a Selic só em parte. A correlação geral entre os dois é 0,57, longe de um movimento colado. E a correlação móvel de 12 meses balança muito, ficando negativa em vários trechos, inclusive agora, perto de zero. Isso quer dizer que existem períodos em que o banco aumenta o spread mesmo com o juro básico parado. O custo do dinheiro para quem pega crédito não é explicado só pela Selic, tem o prêmio de risco que os bancos cobram por conta própria.

## Pergunta 4: quanto tempo a inadimplência leva para reagir aos juros?

Em vez de chutar um número, a query testa várias defasagens de uma vez (de 0 a 18 meses) e mede em qual delas a inadimplência mais se parece com a Selic do passado. A defasagem com a maior correlação é a resposta.

In [ ]:
p4 = rodar("p4_defasagem_otima.sql")
p4_ord = p4.sort_values("meses_de_defasagem").reset_index(drop=True)

idx_pico  = p4_ord["correlacao"].abs().idxmax()
lag_pico  = p4_ord.loc[idx_pico, "meses_de_defasagem"]
corr_pico = p4_ord.loc[idx_pico, "correlacao"]

cores_bars = [COR["laranja"] if m == lag_pico else COR["azul"]
              for m in p4_ord["meses_de_defasagem"]]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(p4_ord["meses_de_defasagem"], p4_ord["correlacao"],
       color=cores_bars, width=0.7)
ax.text(
    0.98, 0.95,
    f"Pico: lag {int(lag_pico)} meses  (r = {corr_pico:.2f})",
    transform=ax.transAxes, ha="right", va="top",
    color=COR["laranja"], fontsize=8.5, fontweight="bold",
)
ax.set_title("Correlação entre Inadimplência e Selic por Defasagem", fontsize=13)
ax.set_xlabel("Defasagem (meses)", fontsize=10)
ax.set_ylabel("Correlação (r)", fontsize=10)
ax.set_xticks(p4_ord["meses_de_defasagem"])
fig.tight_layout()
plt.show()

In [ ]:
p4.head(3)

### Conclusão

Essa é a descoberta mais forte do projeto. A inadimplência total reage à Selic com cerca de 9 a 10 meses de atraso, e a relação é robusta, com correlação de 0,83. Na prática, um aperto nos juros hoje só vira atraso de pagamento perto de três trimestres depois. Quem acompanha risco de crédito ganha aqui um sinal antecipado: dá para estimar a inadimplência de daqui a quase um ano olhando o que a Selic está fazendo agora.

## Pergunta 5: em que fase do ciclo o país está hoje?

A query classifica cada mês combinando a direção da inadimplência e da Selic nos últimos três meses. É uma regra simples e transparente, não um modelo, feita para dar uma leitura rápida do momento.

In [ ]:
p5 = rodar("p5_ciclo_pais.sql")

contagem = p5["fase_do_ciclo"].value_counts().sort_values(ascending=True)

_MAP_COR = {
    "deterioração": COR["vermelho"],
    "aquecimento":  COR["laranja"],
    "estabilidade": COR["cinza"],
    "recuperação":  COR["verde"],
}
cores_fases = [_MAP_COR.get(f.lower(), COR["cinza"]) for f in contagem.index]

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(contagem.index, contagem.values, color=cores_fases, height=0.55)
ax.bar_label(bars, padding=5, fontsize=9, color="#333333", fmt="%d meses")
ax.set_title("Distribuição de Meses por Fase do Ciclo de Crédito", fontsize=13)
ax.set_xlabel("Número de meses", fontsize=10)
ax.set_xlim(right=contagem.max() * 1.15)
fig.tight_layout()
plt.show()

In [ ]:
p5.tail(6)

### Conclusão

Juntando tudo, hoje o país está numa fase de deterioração do crédito. Nos últimos meses a inadimplência total voltou a subir, saindo de cerca de 4,0% para 4,4%, enquanto a Selic segue num patamar alto, perto de 15%. Isso conversa direto com a pergunta 4: o juro alto de 2024 e 2025 está chegando agora na conta da inadimplência. Olhando a série inteira, a deterioração é a fase mais frequente, o que diz bastante sobre como o crédito brasileiro passa mais tempo sob pressão do que folgado.

## Principais conclusões

1. A inadimplência da pessoa física é estruturalmente mais alta e mais volátil que a da pessoa jurídica, e voltou a subir desde o fundo de 2020.
2. As famílias estão mais endividadas e comprometendo uma fatia maior da renda, dois movimentos que andam juntos e atingiram os maiores níveis da série nos últimos anos.
3. O spread bancário só acompanha a Selic em parte, e em vários momentos sobe por conta própria, sinal de prêmio de risco além do juro básico.
4. A inadimplência reage à Selic com cerca de 9 a 10 meses de defasagem, o que permite antecipar o risco de crédito quase um ano à frente.
5. O país está hoje em fase de deterioração do crédito, com a inadimplência subindo na esteira do juro alto dos últimos dois anos.